In [27]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from pathlib import Path

In [28]:
RAW_DATA = Path("../data/raw")

In [29]:
demo_1999 = pd.read_sas(RAW_DATA / "DEMO.xpt", format="xport")
demo_2001 = pd.read_sas(RAW_DATA / "DEMO_B.xpt", format="xport")
demo_2003 = pd.read_sas(RAW_DATA / "DEMO_C.xpt", format="xport")

In [30]:
print("1999-2000:", demo_1999.shape)
print("2001-2002:", demo_2001.shape)
print("2003-2004:", demo_2003.shape)

1999-2000: (9965, 144)
2001-2002: (11039, 37)
2003-2004: (10122, 44)


In [31]:
demo = pd.concat(
    [demo_1999, demo_2001, demo_2003],
    ignore_index=True
)

print("Combined participants:", len(demo))

Combined participants: 31126


In [32]:
age40 = demo.loc[demo["RIDAGEYR"] >= 40].copy()

print("Age 40+:", len(age40))

Age 40+: 9970


In [33]:
MORT_DATA = Path("../data/raw/mortality")

In [34]:
# Columns we need from the mortality files
colspecs = [
    (0, 6),     # SEQN: participant ID
    (14, 15),   # ELIGSTAT: eligible for mortality follow-up
    (15, 16),   # MORTSTAT: 0 = alive, 1 = deceased
    (42, 45),   # PERMTH_INT: months of follow-up
]

names = [
    "SEQN",
    "ELIGSTAT",
    "MORTSTAT",
    "PERMTH_INT",
]

In [35]:
mort_1999 = pd.read_fwf(
    MORT_DATA / "NHANES_1999_2000_MORT_2019_PUBLIC.dat",
    colspecs=colspecs,
    names=names
)

mort_2001 = pd.read_fwf(
    MORT_DATA / "NHANES_2001_2002_MORT_2019_PUBLIC.dat",
    colspecs=colspecs,
    names=names
)

mort_2003 = pd.read_fwf(
    MORT_DATA / "NHANES_2003_2004_MORT_2019_PUBLIC.dat",
    colspecs=colspecs,
    names=names
)

print("1999-2000 mortality:", len(mort_1999))
print("2001-2002 mortality:", len(mort_2001))
print("2003-2004 mortality:", len(mort_2003))

1999-2000 mortality: 9965
2001-2002 mortality: 11039
2003-2004 mortality: 10122


In [36]:
print(mort_1999.head())

   SEQN  ELIGSTAT MORTSTAT PERMTH_INT
0     1         2        .          .
1     2         1        1        177
2     3         2        .          .
3     4         2        .          .
4     5         1        0        244


In [37]:
mortality = pd.concat(
    [mort_1999, mort_2001, mort_2003],
    ignore_index=True
)

print("Combined mortality records:", len(mortality))

Combined mortality records: 31126


In [38]:
cohort = age40.merge(
    mortality,
    on="SEQN",
    how="left"
)

print("Age 40+ after mortality merge:", len(cohort))

Age 40+ after mortality merge: 9970


In [39]:
# Keep only participants eligible for mortality follow-up
final_cohort = cohort.loc[cohort["ELIGSTAT"] == 1].copy()

print("Final cohort:", len(final_cohort))

Final cohort: 9957


In [40]:
clean_data = final_cohort.copy()

In [41]:
# Make sure mortality columns are numeric

clean_data["MORTSTAT"] = pd.to_numeric(
    clean_data["MORTSTAT"],
    errors="coerce"
)

clean_data["PERMTH_INT"] = pd.to_numeric(
    clean_data["PERMTH_INT"],
    errors="coerce"
)

In [42]:
print(clean_data[["MORTSTAT", "PERMTH_INT"]].head())

print("\nMORTSTAT:")
print(clean_data["MORTSTAT"].value_counts(dropna=False))

print("\nMaximum follow-up months:")
print(clean_data["PERMTH_INT"].max())

   MORTSTAT  PERMTH_INT
0         1         177
1         0         244
2         0         237
3         1         231
4         1          16

MORTSTAT:
MORTSTAT
0    5467
1    4490
Name: count, dtype: int64

Maximum follow-up months:
249


In [43]:
# 1 = died within 5 years
# 0 = did not die within 5 years

clean_data["death_5yr"] = (
    (clean_data["MORTSTAT"] == 1) &
    (clean_data["PERMTH_INT"] <= 60)
).astype(int)

In [44]:
print(clean_data["death_5yr"].value_counts())

print("\nPercent:")
print(clean_data["death_5yr"].value_counts(normalize=True) * 100)

death_5yr
0    8664
1    1293
Name: count, dtype: int64

Percent:
death_5yr
0    87.014161
1    12.985839
Name: proportion, dtype: float64


In [45]:
# Save the finished 9,957-person cohort
# so the next notebook can start from here

clean_data.to_csv(
    "../data/processed/cohort_9957.csv",
    index=False
)

print("Saved:", clean_data.shape)

Saved: (9957, 158)
